# SQL Queries: NYC Collisions

**Author:** Yougi Jain
**Project:** NYC-Collisions

Runs each query in `sql/` against the published dataset through DuckDB and
checks it returns rows. The dataset is resolved by `app/db.py`: a local
build if present, otherwise the published Release asset, otherwise the
committed seed.


### Setup

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

sys.path.insert(0, str(Path("..").resolve() / "app"))
import db

connection = db.connect()

bounds = db.query(connection, "00_bounds.sql").iloc[0]
print(f"{int(bounds['total_crashes']):,} crashes")
print(f"{bounds['min_datetime']} to {bounds['max_datetime']}")
print("boroughs:", list(bounds["boroughs"]))

### Filter parameters

Every query except `00_bounds.sql` is parameterised on a date window and a
borough list, so the notebook binds the full range here. Narrow these to
explore a slice.

In [ ]:
params = db.filter_params(
    start_date=bounds["min_datetime"],
    end_date=bounds["max_datetime"],
    boroughs=list(bounds["boroughs"]),
    row_limit=5_000,
)
params

## SQL script validation

In [ ]:
sql_files = sorted(p.name for p in Path("../sql").glob("*.sql"))

for name in sql_files:
    display(Markdown(f"## `{name}`"))
    display(Markdown(f"```sql\n{db.read_sql_file(name)}\n```"))

    result = db.query(connection, name, params)
    assert not result.empty, f"`{name}` returned no rows"
    display(result.head(7))

In [ ]:
connection.close()